In [14]:
!pip install -qq pytabkit

In [15]:
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [16]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/test.csv")

train.columns = train.columns.str.replace(" ", "_")
test.columns  = test.columns.str.replace(" ", "_")

target = "Heart_Disease"
id_col = "id"

train[target] = train[target].map({"Presence":1, "Absence":0})

X = train.drop([target, id_col], axis=1)
y = train[target]
X_test = test.drop(id_col, axis=1)

In [17]:
X_xgb = pd.get_dummies(X, drop_first=True)
X_test_xgb = pd.get_dummies(X_test, drop_first=True)

X_xgb, X_test_xgb = X_xgb.align(X_test_xgb, join="left", axis=1, fill_value=0)
cat_cols = X.select_dtypes(include="object").columns.tolist()

In [18]:
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",    
    "device": "cuda",           
    "learning_rate": 0.06053371556553135,
    "max_depth": 3,
    "min_child_weight": 6,
    "subsample": 0.7652061652542972,
    "colsample_bytree": 0.7809576726700508,
    "reg_alpha": 2.9475314124829888,
    "reg_lambda": 0.008472874114482,
    "seed": RANDOM_STATE
}

In [19]:
cat_params = {
    "iterations": 6000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5,
    "eval_metric": "AUC",
    "random_seed": RANDOM_STATE,
    "task_type": "GPU",
    "verbose": False
}

In [20]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

pred_xgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

seeds = [42, 2025, 999]   # Multi-seed stabilization

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):

    # =====================
    # --- XGB MULTI-SEED ---
    # =====================
    dtrain = xgb.DMatrix(X_xgb.iloc[tr_idx], label=y.iloc[tr_idx])
    dval   = xgb.DMatrix(X_xgb.iloc[val_idx], label=y.iloc[val_idx])
    dtest  = xgb.DMatrix(X_test_xgb)

    fold_pred_val = np.zeros(len(val_idx))
    fold_pred_test = np.zeros(len(X_test))

    for seed in seeds:
        params = xgb_params.copy()
        params["seed"] = seed
        params["tree_method"] = "hist"
        params["device"] = "cuda"

        model_xgb = xgb.train(
            params,
            dtrain,
            num_boost_round=4000,
            evals=[(dval, "val")],
            early_stopping_rounds=300,
            verbose_eval=False
        )

        fold_pred_val += model_xgb.predict(dval) / len(seeds)
        fold_pred_test += model_xgb.predict(dtest) / len(seeds)

    oof_xgb[val_idx] = fold_pred_val
    pred_xgb += fold_pred_test / 5


    # =====================
    # --- CatBoost ---
    # =====================
    model_cat = CatBoostClassifier(**cat_params)

    model_cat.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
        cat_features=cat_cols,
        early_stopping_rounds=300,
        verbose=False
    )

    oof_cat[val_idx] = model_cat.predict_proba(X.iloc[val_idx])[:,1]
    pred_cat += model_cat.predict_proba(X_test)[:,1] / 5

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


In [21]:
print("XGB OOF:", roc_auc_score(y, oof_xgb))
print("CAT OOF:", roc_auc_score(y, oof_cat))

XGB OOF: 0.9555316611046967
CAT OOF: 0.9553540592313368


In [22]:
# =====================
# --- Target Encoded XGB ---
# =====================

from sklearn.preprocessing import TargetEncoder

oof_te_xgb = np.zeros(len(X))
pred_te_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    
    X_tr = X.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx]
    X_te = X_test.copy()
    
    for col in X.columns:
        TE = TargetEncoder(cv=5, random_state=RANDOM_STATE)
        X_tr[[col]] = TE.fit_transform(X_tr[[col]], y_tr)
        X_val[[col]] = TE.transform(X_val[[col]])
        X_te[[col]] = TE.transform(X_te[[col]])
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)
    dtest  = xgb.DMatrix(X_te)

    model_te = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=4000,
        evals=[(dval, "val")],
        early_stopping_rounds=300,
        verbose_eval=False
    )
    
    oof_te_xgb[val_idx] = model_te.predict(dval)
    pred_te_xgb += model_te.predict(dtest) / 5

print("TE-XGB OOF:", roc_auc_score(y, oof_te_xgb))

TE-XGB OOF: 0.9555525144760162


In [23]:
# =====================
# --- RealMLP on X_xgb ---
# =====================

oof_mlp = np.zeros(len(X))
pred_mlp = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_xgb, y)):

    X_tr = X_xgb.iloc[tr_idx].copy()
    y_tr = y.iloc[tr_idx]
    X_val = X_xgb.iloc[val_idx].copy()
    y_val = y.iloc[val_idx]
    X_te = X_test_xgb.copy()

    model_mlp = RealMLP_TD_Classifier(
        n_epochs=20,
        batch_size=2048,
        device='cuda',
        verbosity=0,
        val_metric_name='1-auc_ovr'
    )

    model_mlp.fit(
        X_tr, y_tr,
        X_val, y_val
    )

    oof_mlp[val_idx] = model_mlp.predict_proba(X_val)[:,1]
    pred_mlp += model_mlp.predict_proba(X_te)[:,1] / 5

print("MLP OOF:", roc_auc_score(y, oof_mlp))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=20` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics a

MLP OOF: 0.9534112990814548


In [24]:
# =====================
# FIND BEST 4-MODEL BLEND
# =====================

best_score = 0
best_w = None

for w_xgb in np.arange(0.75, 0.95, 0.02):
    for w_cat in np.arange(0.0, 0.15, 0.02):
        for w_te in np.arange(0.0, 0.15, 0.02):
            
            w_mlp = 1 - w_xgb - w_cat - w_te
            if w_mlp < 0:
                continue
            
            blend = (
                w_xgb * oof_xgb +
                w_cat * oof_cat +
                w_te  * oof_te_xgb +
                w_mlp * oof_mlp
            )
            
            score = roc_auc_score(y, blend)
            
            if score > best_score:
                best_score = score
                best_w = (w_xgb, w_cat, w_te, w_mlp)

print("Best weights (XGB, CAT, TE, MLP):", best_w)
print("Best OOF:", best_score)


# =====================
# FINAL RANK BLEND ON TEST
# =====================

from scipy.stats import rankdata

rank_xgb = rankdata(pred_xgb)
rank_cat = rankdata(pred_cat)
rank_te  = rankdata(pred_te_xgb)
rank_mlp = rankdata(pred_mlp)

w_xgb, w_cat, w_te, w_mlp = best_w

final_rank = (
    w_xgb * rank_xgb +
    w_cat * rank_cat +
    w_te  * rank_te +
    w_mlp * rank_mlp
)

final_pred = final_rank / final_rank.max()

Best weights (XGB, CAT, TE, MLP): (np.float64(0.75), np.float64(0.1), np.float64(0.14), np.float64(0.009999999999999981))
Best OOF: 0.9555811252519173


In [25]:
submission = pd.DataFrame({
    id_col: test[id_col],
    target: final_pred
})

submission.to_csv("submission_realmlp_v3.csv", index=False)
submission.head()

,id,Heart_Disease
0,630000,0.787588
1,630001,0.076808
2,630002,0.904584
3,630003,0.040339
4,630004,0.449749
